# Machine Learning with scikit-learn

**scikit-learn** is Python's most popular machine learning library. It provides tools for classification, regression, clustering, and more.

Install: `pip install scikit-learn numpy pandas`

### ML Workflow
1. Get data
2. Preprocess (clean, encode, scale)
3. Split into train/test
4. Choose and train a model
5. Evaluate performance
6. Improve (tune hyperparameters, feature engineering)

In [ ]:
import sklearn
import numpy as np

print(f'scikit-learn: {sklearn.__version__}')
print(f'numpy: {np.__version__}')

## 1. Core Concepts: Supervised vs Unsupervised

| Type | Goal | Examples |
|------|------|----------|
| **Supervised — Classification** | Predict a category | spam/not-spam, iris species |
| **Supervised — Regression** | Predict a number | house price, temperature |
| **Unsupervised — Clustering** | Find groups | customer segments |
| **Unsupervised — Dimensionality reduction** | Simplify | PCA |

## 2. The Iris Dataset — Classification

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

# Load dataset
iris = load_iris()
X = iris.data      # features: sepal length, sepal width, petal length, petal width
y = iris.target    # labels: 0=setosa, 1=versicolor, 2=virginica

print(f'Samples: {X.shape[0]}')
print(f'Features: {X.shape[1]}')
print(f'Classes: {iris.target_names}')
print(f'\nFirst 3 samples:')
for i in range(3):
    print(f'  {X[i]} → {iris.target_names[y[i]]}')

In [ ]:
# Split into train (80%) and test (20%)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Training samples: {len(X_train)}')
print(f'Test samples: {len(X_test)}')

# Scale features (important for many algorithms)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)  # fit AND transform
X_test_scaled = scaler.transform(X_test)        # transform ONLY (use train stats)

print(f'\nOriginal mean: {X_train[:, 0].mean():.2f}')
print(f'Scaled mean:   {X_train_scaled[:, 0].mean():.2f}')

## 3. Training Models

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score

# All scikit-learn models share the same API: fit() and predict()
models = {
    'Logistic Regression': LogisticRegression(random_state=42, max_iter=200),
    'Decision Tree': DecisionTreeClassifier(random_state=42),
    'K-Nearest Neighbors': KNeighborsClassifier(n_neighbors=5),
    'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42),
}

results = {}
for name, model in models.items():
    model.fit(X_train_scaled, y_train)           # TRAIN
    y_pred = model.predict(X_test_scaled)        # PREDICT
    accuracy = accuracy_score(y_test, y_pred)    # EVALUATE
    results[name] = accuracy
    print(f'{name:25}: {accuracy:.1%}')

best = max(results, key=results.get)
print(f'\nBest model: {best} ({results[best]:.1%})')

## 4. Detailed Evaluation

In [ ]:
from sklearn.metrics import classification_report, confusion_matrix

# Use the best model (Random Forest)
rf = RandomForestClassifier(n_estimators=100, random_state=42)
rf.fit(X_train_scaled, y_train)
y_pred = rf.predict(X_test_scaled)

# Detailed metrics
print('=== Classification Report ===')
print(classification_report(y_test, y_pred, target_names=iris.target_names))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
print('=== Confusion Matrix ===')
print(f'          {" ".join(iris.target_names)}')
for i, row in enumerate(cm):
    label = iris.target_names[i]
    print(f'{label:10} {row}')

## 5. Regression — Predicting a Number

In [ ]:
from sklearn.datasets import fetch_california_housing
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score
import numpy as np

# Load housing data
housing = fetch_california_housing()
X, y = housing.data[:1000], housing.target[:1000]  # use subset for speed

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

scaler2 = StandardScaler()
X_train_s = scaler2.fit_transform(X_train)
X_test_s = scaler2.transform(X_test)

# Train models
for name, model in [('Linear Regression', LinearRegression()),
                     ('Gradient Boosting', GradientBoostingRegressor(random_state=42))]:
    model.fit(X_train_s, y_train)
    y_pred = model.predict(X_test_s)
    rmse = mean_squared_error(y_test, y_pred, squared=False)
    r2 = r2_score(y_test, y_pred)
    print(f'{name}: RMSE={rmse:.3f}, R²={r2:.3f}')

## 6. Pipeline — Combine Steps

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC

# Pipeline chains preprocessing + model — no data leakage!
pipe = Pipeline([
    ('scaler', StandardScaler()),
    ('classifier', SVC(kernel='rbf', random_state=42))
])

# Works like any model
X_train2, X_test2, y_train2, y_test2 = train_test_split(X, y[:len(X)], test_size=0.2, random_state=42)

iris = load_iris()
pipe.fit(iris.data[:120], iris.target[:120])
y_pred = pipe.predict(iris.data[120:])
print(f'SVM Pipeline accuracy: {accuracy_score(iris.target[120:], y_pred):.1%}')

## Mini-Project: Iris Flower Classifier

In [ ]:
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.metrics import classification_report, accuracy_score
import numpy as np

print('=== IRIS FLOWER CLASSIFIER ===')
print()

# 1. Load data
iris = load_iris()
X, y = iris.data, iris.target
print(f'Dataset: {X.shape[0]} samples, {X.shape[1]} features')
print(f'Classes: {dict(enumerate(iris.target_names))}')

# 2. Explore
print('\nClass distribution:')
for cls_id, cls_name in enumerate(iris.target_names):
    count = (y == cls_id).sum()
    print(f'  {cls_name}: {count} samples')

# 3. Split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

# 4. Build pipeline
pipeline = Pipeline([
    ('scaler', StandardScaler()),
    ('clf', RandomForestClassifier(n_estimators=100, random_state=42))
])

# 5. Cross-validation
cv_scores = cross_val_score(pipeline, X_train, y_train, cv=5)
print(f'\nCross-validation (5-fold):')
print(f'  Scores: {[f"{s:.1%}" for s in cv_scores]}')
print(f'  Mean: {cv_scores.mean():.1%} ± {cv_scores.std():.1%}')

# 6. Train and evaluate
pipeline.fit(X_train, y_train)
y_pred = pipeline.predict(X_test)

print(f'\nTest set accuracy: {accuracy_score(y_test, y_pred):.1%}')
print()
print(classification_report(y_test, y_pred, target_names=iris.target_names))

# 7. Predict new samples
new_samples = np.array([
    [5.1, 3.5, 1.4, 0.2],  # likely setosa
    [6.7, 3.1, 4.7, 1.5],  # likely versicolor
    [6.3, 3.3, 6.0, 2.5],  # likely virginica
])

predictions = pipeline.predict(new_samples)
probabilities = pipeline.predict_proba(new_samples)

print('Predictions for new samples:')
for i, (pred, probs) in enumerate(zip(predictions, probabilities)):
    name = iris.target_names[pred]
    confidence = probs.max()
    print(f'  Sample {i+1}: {name} ({confidence:.1%} confidence)')

## Feature Importance

In [ ]:
# Feature importance from Random Forest
rf_clf = pipeline.named_steps['clf']
importances = rf_clf.feature_importances_

print('Feature Importances:')
for name, importance in sorted(zip(iris.feature_names, importances), key=lambda x: -x[1]):
    bar = '#' * int(importance * 40)
    print(f'  {name:25} {importance:.3f}  {bar}')